# Stap 4: PID-regeling vanuit Python

Upload eerst `arduino/Stap_4_PID_Python_aansturing/Stap_4_PID_Python_aansturing.ino`. Sluit de Serial Monitor en Serial Plotter. De Arduino voert de PID-regeling uit; Python stelt de parameters in en slaat de meting op.

In [3]:
from closed_loop.meting import test_hoek, controleer_sensor, voer_pid_meting_uit
import matplotlib.pyplot as plt
com_poort = "COM3"  # Vul de poort van jouw Arduino in.

### 1. Beginhoek meten

Laat de motorvoeding uit en zet de arm in de nulstand. De hoektest toont gedurende `meetduur` seconden de tijd en ruwe sensorhoek. `delta_t` is het gewenste aanvraaginterval; tijdstippen worden bij ontvangst in Python bepaald. Neem de hoek in de nulstand over als `beginhoek`. De werkhoek wordt beginhoek - sensorhoek, omgerekend naar -180..180 graden.


In [4]:
hoekmetingen = test_hoek(com_poort=com_poort, meetduur=10.0, delta_t=0.1)


tijd_s,hoek_deg


RuntimeError: Arduino heeft de meting gestopt: ERROR:MAGNET_STATUS:MD=1;ML=1;MH=0

### 2. Instellen en meten

Begin met P: `ki = 0` en `kd = 0`. Kies met de begeleider een haalbare doelhoek, versterking en PWM-grens op basis van open loop. De waarden hieronder zijn beginwaarden. `max_pwm` begrenst de PID-uitgang; de regelaar kiest de actuele PWM. `delta_t` bepaalt het meet- en PID-interval. `fixed` stopt na `meetduur`; `continuous` loopt tot een celonderbreking of fout. Verbindingsbewaking en veiligheidscontroles blijven actief. Schakel daarna de gecontroleerde motorvoeding in. Deze cel start de regeling direct na de communicatiecontrole, zonder stabilisatietijd. Onderbreek de cel om eerder te stoppen.

In [9]:
beginhoek = 90.0  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 40.0    # graden ten opzichte van de ingestelde nulstand
kp = 2.0
ki = 0.0
kd = 0.0
max_pwm = 120     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz
mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)

metingen, csv_pad = voer_pid_meting_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm,
    meetduur=meetduur, delta_t=delta_t, mode=mode,
)

RuntimeError: Arduino heeft de meting gestopt: ERROR:MAGNET_STATUS

### 3. Resultaat bekijken

Meetpunten staan in `data/` als CSV: eerst drie regels instellingen, daarna `tijd_s,hoek_deg` en de meetpunten. Inlezen kan met `pd.read_csv(csv_pad, skiprows=3)`; een JSON-bestand met dezelfde naam bewaart de gekozen instellingen en de afloopstatus. Verander voor PI alleen `ki`; onderzoek daarna voor PID ook `kd`. Laat de arm voor iedere proef tot rust komen.

In [ ]:
if metingen:
    tijden, doelen, hoeken, pwm_waarden = zip(*metingen)
    fig, ax = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
    ax[0].plot(tijden, hoeken, label="Gemeten hoek")
    ax[0].plot(tijden, doelen, "--", label="Doelhoek")
    ax[0].set_ylabel("Hoek (graden)")
    ax[0].legend()
    ax[1].plot(tijden, pwm_waarden)
    ax[1].set_ylabel("PWM")
    ax[1].set_xlabel("Tijd sinds start (s)")
    for a in ax:
        a.grid()
    fig.tight_layout()
    plt.show()